In [12]:
import fastf1
from fastf1 import Cache

Cache.enable_cache('cache')
Cache.enable_cache(
    cache_dir='cache',
    ignore_version=False,
    force_renew=False,
    use_requests_cache=True
)
schedule = fastf1.get_event_schedule(2025)
schedule = schedule[['RoundNumber', 'EventDate', 'EventName']]
print(schedule)

    RoundNumber  EventDate                  EventName
0             0 2025-02-28         Pre-Season Testing
1             1 2025-03-16      Australian Grand Prix
2             2 2025-03-23         Chinese Grand Prix
3             3 2025-04-06        Japanese Grand Prix
4             4 2025-04-13         Bahrain Grand Prix
5             5 2025-04-20   Saudi Arabian Grand Prix
6             6 2025-05-04           Miami Grand Prix
7             7 2025-05-18  Emilia Romagna Grand Prix
8             8 2025-05-25          Monaco Grand Prix
9             9 2025-06-01         Spanish Grand Prix
10           10 2025-06-15        Canadian Grand Prix
11           11 2025-06-29        Austrian Grand Prix
12           12 2025-07-06         British Grand Prix
13           13 2025-07-27         Belgian Grand Prix
14           14 2025-08-03       Hungarian Grand Prix
15           15 2025-08-31           Dutch Grand Prix
16           16 2025-09-07         Italian Grand Prix
17           17 2025-09-21  

In [ ]:
import pandas as pd
import numpy as np


def build_season(year):
    schedule = fastf1.get_event_schedule(year)
    schedule = schedule[schedule['RoundNumber'] > 0]  

    season_rows = []
    for rnd in sched['RoundNumber']:
        rnd = int(rnd)
        event_name = sched.loc[sched['RoundNumber'] == rnd, 'EventName'].values[0]
        try:
            session = fastf1.get_session(year, rnd, 'R')
            session.load(laps=True, weather=True)
        except Exception as e:
            print(f"Skipping {year} round {rnd} ({event_name}): {e}")
            continue

        # Race results already carry starting grid + finishing position
        df = session.results.copy()
        df['Year'] = year
        df['Round'] = rnd
        df['EventName'] = event_name
        df = df.rename(columns={
            'GridPosition': 'StartingGridPosition',
            'Position': 'FinishingPosition',
        })

        # --- Lap time per driver (seconds) ---
        laps = session.laps
        if laps is not None and not laps.empty:
            lap_agg = laps.groupby('Driver')['LapTime'].agg(
                AvgLapTime='mean',
                MedianLapTime='median',
                FastestLapTime='min',
            ).reset_index()
            for col in ['AvgLapTime', 'MedianLapTime', 'FastestLapTime']:
                lap_agg[col] = lap_agg[col].dt.total_seconds()
            df = df.merge(lap_agg, left_on='Abbreviation', right_on='Driver', how='left')
            df = df.drop(columns=['Driver'])

        # --- Weather (session-level; same value for every driver at this track) ---
        weather = session.weather_data
        if weather is not None and not weather.empty:
            df['AirTemp'] = weather['AirTemp'].mean()
            df['TrackTemp'] = weather['TrackTemp'].mean()
            df['Humidity'] = weather['Humidity'].mean()
            df['Pressure'] = weather['Pressure'].mean()
            df['WindSpeed'] = weather['WindSpeed'].mean()
            df['Rainfall'] = bool(weather['Rainfall'].any())

        season_rows.append(df)

    season_df = pd.concat(season_rows, ignore_index=True)

    keep = [
        'Year', 'Round', 'EventName',
        'Abbreviation', 'DriverNumber', 'TeamName',
        'StartingGridPosition', 'FinishingPosition',
        'AvgLapTime', 'MedianLapTime', 'FastestLapTime',
        'AirTemp', 'TrackTemp', 'Humidity', 'Pressure', 'WindSpeed', 'Rainfall',
    ]
    keep = [c for c in keep if c in season_df.columns]
    return season_df[keep]


In [ ]:
# Build 2024: weather, lap time, starting grid + finishing position per driver per track
results_2024_df = build_season(2024)
print(results_2024_df.shape)
results_2024_df.head(20)


In [ ]:
# Build 2025: weather, lap time, starting grid + finishing position per driver per track
results_2025_df = build_season(2025)
print(results_2025_df.shape)
results_2025_df.head(20)


In [16]:
import openpyxl as xl
results_2024_df.to_excel('f1_2024_results.xlsx', index=False)
#raw data

In [ ]:
results_2025_df.to_excel('f1_results_2025.xlsx', index=False)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler()
df['scaled_lap_time'] = scaler.fit_transform(df[['age']])